# DNC Notebook

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from types import SimpleNamespace

# Assurez-vous que ces imports fonctionnent selon votre structure de dossier
from problems.problem_cvrp import CVRP, get_capacity
from agent.dnc import DNC # Ou le fichier où vous avez mis la classe DNC
from utils.utils import move_to # Fonction utilitaire classique de NeuOpt
from options import get_options


import plotly.graph_objects as go


In [7]:
opts = get_options('')
opts.problem = 'cvrp' 
opts.wo_feature1 = False #feature1 corresponds to these two features : infeasibility_indicator_before_visit,infeasibility_indicator_after_visit. If wo_feature1 is False, then agent has those two features. 
opts.wo_feature3 = False #feature3 corresponds to exploration statistics (i think).
opts.wo_regular = False
opts.wo_bonus = False
opts.wo_RNN = False
opts.wo_MDP = True
opts.use_cuda = True
opts.stall_limit = 10
opts.val_m = 1
opts.graph_size = 400
opts.init_val_met = 'random'
opts.no_saving = True
opts.no_tb = True
opts.val_size = 10
opts.load_path = 'pre-trained/cvrp100.pt'
opts.device = torch.device("cuda" if opts.use_cuda else "cpu")
opts.no_progress_bar = True
opts.batch_size = 2
opts

print(f"Utilisation du device : {opts.device}")

Utilisation du device : cuda


In [8]:
global_problem = CVRP(
                        p_size = opts.graph_size,
                        init_val_met = opts.init_val_met,
                        with_assert = opts.use_assert,
                        DUMMY_RATE = opts.dummy_rate,
                        k = opts.k,
                        with_bonus = not opts.wo_bonus,
                        with_regular = not opts.wo_regular
                        )

agent = DNC(global_problem, opts)
checkpoint_path = 'pre-trained/cvrp100.pt'
agent.load(checkpoint_path)


CVRP with 400 nodes and 200 dummy depots (total 600).
 Regulation: True Bonus: True Do assert: False.
 MAX 4-opt.

CVRP with 100 nodes and 50 dummy depots (total 150).
 Regulation: True Bonus: True Do assert: False.
 MAX 4-opt.

simpleMDP:  True
# params in Actor {'Total': 685140, 'Trainable': 685140}
 [*] Loading data from pre-trained/cvrp100.pt


In [15]:
print(f"Génération d'une instance de taille {opts.graph_size}...")
    
    # On crée un batch manuellement via la classe problem
    # Cela évite de devoir charger un fichier .pkl externe
dummy_size = int(opts.graph_size * opts.dummy_rate) # 200
total_size = opts.graph_size + dummy_size           # 600
print(opts.batch_size)

coords = torch.cat((
    torch.FloatTensor(opts.batch_size, 1, 2).uniform_(0, 1).repeat(1, dummy_size, 1),
    torch.FloatTensor(opts.batch_size, global_problem.real_size, 2).uniform_(0, 1) 
), dim=1)

demand = torch.cat((
    torch.zeros(opts.batch_size, dummy_size),
    torch.FloatTensor(opts.batch_size, global_problem.real_size).uniform_(1, 10).long().float() / get_capacity(global_problem.real_size)
), dim=1)
batch = {
    'coordinates': coords,
    'demand': demand
}
print(batch['coordinates'].shape)

Génération d'une instance de taille 400...
2
torch.Size([2, 600, 2])


In [5]:
def plot_dnc(batch, results, n_splits, idx=0):
    """
    Visualisation interactive sans les lignes retournant au dépôt.
    Affiche uniquement les connexions inter-clients.
    """
    
    # 1. Extraction des données
    coords = batch['coordinates'][idx].cpu().numpy()
    full_route = results['routes'][idx].cpu().numpy()
    cost = results['total_cost'][idx].item()
    depot_pos = coords[0]

    # 2. Initialisation Figure
    fig = go.Figure()

    # --- A. Clients (Fond) ---
    fig.add_trace(go.Scatter(
        x=coords[1:, 0], y=coords[1:, 1],
        mode='markers',
        marker=dict(size=4, color='lightgray'),
        name='Clients',
        hoverinfo='skip'
    ))

    # --- B. Dépôt ---
    fig.add_trace(go.Scatter(
        x=[depot_pos[0]], y=[depot_pos[1]],
        mode='markers',
        marker=dict(symbol='square', size=12, color='red', line=dict(width=1, color='black')),
        name='Dépôt',
        hoverinfo='name'
    ))

    # --- C. Tracé par Split (Sans retour dépôt) ---
    total_steps = len(full_route)
    steps_per_split = total_steps // n_splits
    
    colors = [
        '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', 
        '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf'
    ]

    for i in range(n_splits):
        start_idx = i * steps_per_split
        end_idx = (i + 1) * steps_per_split
        segment_indices = full_route[start_idx:end_idx]
        
        # --- LOGIQUE DE NETTOYAGE ---
        x_vals = []
        y_vals = []
        hover_texts = []
        
        for node_idx in segment_indices:
            if node_idx == 0:
                # C'est le dépôt (ou un dummy) : On coupe la ligne !
                # Insérer None dans Plotly crée une rupture visuelle
                x_vals.append(None)
                y_vals.append(None)
                hover_texts.append(None)
            else:
                # C'est un client : On ajoute le point
                c = coords[node_idx]
                x_vals.append(c[0])
                y_vals.append(c[1])
                hover_texts.append(str(node_idx))
        
        # Tracé
        split_name = f"Split {i+1}"
        color = colors[i % len(colors)]
        
        fig.add_trace(go.Scatter(
            x=x_vals,
            y=y_vals,
            mode='lines+markers', # Les points isolés (sans voisin client) apparaitront quand même
            line=dict(width=2, color=color),
            marker=dict(size=6, color=color),
            name=split_name,
            legendgroup=split_name,
            text=hover_texts,
            hovertemplate=f"<b>{split_name}</b><br>Client: %{{text}}<extra></extra>",
            connectgaps=False # Important : ne pas relier par dessus les None
        ))

    # 3. Layout
    fig.update_layout(
        title=f"<b>Solution DNC (Vue locale)</b> | Coût: {cost:.2f}",
        xaxis=dict(range=[0, 1], scaleanchor="y", scaleratio=1, showgrid=False, zeroline=False),
        yaxis=dict(range=[0, 1], showgrid=False, zeroline=False),
        width=900, height=800,
        legend=dict(itemclick="toggleothers", itemdoubleclick="toggle"),
        template="plotly_white",
        plot_bgcolor='rgba(245,245,245,0.3)' # Fond très léger
    )

    fig.show()

In [16]:
print(f"Lancement de l'inférence (Split en {opts.dnc_n_splits} sous-problèmes)...")
    
# On appelle la méthode solve que nous avons ajoutée à la classe DNC
# T=100 ou 200 itérations de PPO pour raffiner la solution

results = agent.solve(batch, T=500, val_m=opts.val_m, stall_limit=opts.stall_limit)
    

# ==========================================
# 5. RÉSULTATS & VISUALISATION
# ==========================================
final_cost = results['total_cost'][0].item()
print(f"\n🏆 Coût Final Reconstruit : {final_cost:.4f}")

# Visualisation
plot_dnc(batch, results, n_splits=opts.dnc_n_splits, idx=0)


Lancement de l'inférence (Split en 4 sous-problèmes)...

🏆 Coût Final Reconstruit : 37.9958


In [17]:
def check_solution_validity(batch, results, dummy_size):
    """
    Vérifie si les routes reconstruites passent par TOUS les clients réels
    et s'il n'y a pas de doublons.
    
    Args:
        batch (dict): Batch original contenant 'coordinates'.
        results (dict): Résultat de agent.solve() contenant 'routes'.
        dummy_size (int): Nombre de dépôts fictifs (les indices < dummy_size sont ignorés).
        
    Returns:
        bool: True si tout le batch est valide, False sinon.
    """
    routes = results['routes'] # [B, Total_Len]
    print(routes.shape)
    print(batch['coordinates'].shape)
    
    bs = routes.size(0)
    total_size = batch['coordinates'].size(1)
    
    # Les indices des clients réels vont de dummy_size à total_size-1
    expected_clients = set(range(dummy_size, total_size))
    
    all_valid = True
    
    print(f"--- Vérification de la validité (Batch size: {bs}) ---")
    
    for i in range(bs):
        # 1. Extraction de la route (CPU numpy pour manipulation facile des sets)
        route = routes[i].cpu().numpy()
        
        # 2. Filtrage : on ne garde que les indices des clients réels
        # Les indices < dummy_size sont des dépôts (ou dummies), on s'en fiche pour la couverture
        real_visited_list = [idx for idx in route if idx >= dummy_size]
        real_visited_set = set(real_visited_list)
        
        # 3. Vérifications
        is_complete = (real_visited_set == expected_clients)
        has_no_duplicates = (len(real_visited_list) == len(real_visited_set))
        
        if not is_complete:
            missing = expected_clients - real_visited_set
            print(f"❌ Instance {i}: Manque {len(missing)} clients ! (Ex: {list(missing)[:5]}...)")
            all_valid = False
            
        elif not has_no_duplicates:
            print(f"⚠️ Instance {i}: Tous les clients sont là, mais il y a des doublons (Visités {len(real_visited_list)} vs Uniques {len(real_visited_set)})")
            # C'est moins grave que 'manquant', mais c'est souvent sous-optimal
            all_valid = False
            
        else:
            # Optionnel : décommenter pour voir les succès
            # print(f"✅ Instance {i}: Valide ({len(real_visited_set)} clients uniques).")
            pass

    if all_valid:
        print(f"🎉 SUCCÈS : Toutes les instances ({bs}) visitent exactement tous les clients.")
    else:
        print("❌ ÉCHEC : Certaines routes sont invalides.")
        
    return all_valid

In [18]:
check_solution_validity(batch, results, dummy_size=dummy_size)

torch.Size([2, 600])
torch.Size([2, 600, 2])
--- Vérification de la validité (Batch size: 2) ---
🎉 SUCCÈS : Toutes les instances (2) visitent exactement tous les clients.


True